#  Text-to-Speech (TTS) system with Fun-CosyVoice 3.0 and OpenVINO

Fun-CosyVoice 3.0 is an advanced text-to-speech (TTS) system based on large language models (LLM), surpassing its predecessor (CosyVoice 2.0) in content consistency, speaker similarity, and prosody naturalness. It is designed for zero-shot multilingual speech synthesis in the wild.

More details can be found in the original [repository](https://github.com/FunAudioLLM/CosyVoice.git) and [model card](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512)

In this tutorial we consider how to run and optimize Fun-CosyVoice 3.0 using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run zero_shot inference](#Run-zero_shot-inference)
    - [Run cross_lingual inference with fine-grained control](#Run-cross_lingual-inference-with-fine-grained-control)
    - [Run instruct2 inference (fast speed)](#Run-instruct2-inference-fast-speed)
    - [Run hotfix inference (fast speed)](#Run-hotfix-inference-fast-speed)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/cosyvoice3-tts/cosyvoice3-tts.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("cosyvoice3-tts.ipynb")

In [ ]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform
import shutil
import subprocess

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "setuptools<82",
    "nncf",
    "openvino>=2025.4.0",
    "gradio",
    "huggingface_hub",
)

repo_dir = Path("CosyVoice")
revision = "8b54619760fcb78abf5e4637a88e19c1b9ab53c9"
clone_repo("https://github.com/FunAudioLLM/CosyVoice.git", revision)

# Replace CosyVoice requirements.txt with local version
shutil.copy("requirements_cosyvoice.txt", repo_dir / "requirements.txt")

pip_install(
    "-q -r",
    str(repo_dir / "requirements.txt"),
)
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], cwd=repo_dir, check=True)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)


The CosyVoice3 pipeline consists of six important parts:

* **Text Embeddings**: Converts text tokens into embeddings for the language model.
* **Speech Embeddings**: Extracts speaker embeddings from reference audio for voice cloning.
* **Language Model (LLM)**: A Qwen2-based large language model that generates speech tokens from text and speaker conditions.
* **Flow Embeddings**: Embeds speech tokens for the flow matching model.
* **Flow Estimator**: A conditional flow matching (CFM) model that generates mel-spectrogram from speech token embeddings.
* **HiFT (HiFi-GAN based vocoder)**: Converts mel-spectrogram to high-fidelity audio waveform.

The model uses a two-stage approach: first, the LLM generates discrete speech tokens conditioned on text and speaker information, then the flow matching model and vocoder convert these tokens into natural speech audio.

In [ ]:
# Disable torchao to avoid version conflict with torch 2.3.1
import os

os.environ["TRANSFORMERS_NO_TORCHAO"] = "1"

from ov_cosyvoice_helper import convert_cosyvoice
from huggingface_hub import snapshot_download

pt_model_path = "pretrained_models"
snapshot_download(repo_id="FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=Path(pt_model_path))

model_path = "Fun-CosyVoice3-0.5B-2512-ov"
convert_cosyvoice(pt_model_path, model_path)

2026-01-04 02:35:57,305 DEBUG Starting new HTTPS connection (1): huggingface.co:443


failed to import ttsfrd, use wetext instead


2026-01-04 02:35:57,477 DEBUG https://huggingface.co:443 "GET /api/models/FunAudioLLM/Fun-CosyVoice3-0.5B-2512/revision/main HTTP/1.1" 200 1554
2026-01-04 02:35:57,483 DEBUG Attempting to acquire lock 140503711866144 on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2026-01-04 02:35:57,487 DEBUG Lock 140503711866144 acquired on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2026-01-04 02:35:57,488 DEBUG Attempting to acquire lock 140503711870608 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/generation_config.json.lock
2026-01-04 02:35:57,489 DEBUG Attempting to acquire lock 140503711871424 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/model.safetensors.lock
2026-01-04 02:35:57,489 DEBUG Attempting to release lock 140503711866144 on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2026-01-04 02:35:57,490 DEBUG Attempting to acquire lock 140503711873344 on pretrained_models/.cache/huggingface

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

2026-01-04 02:35:57,491 DEBUG Attempting to acquire lock 140503711874400 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/config.json.lock
2026-01-04 02:35:57,492 DEBUG Lock 140503711870608 acquired on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/generation_config.json.lock
2026-01-04 02:35:57,493 DEBUG Attempting to acquire lock 140503711747184 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/vocab.json.lock
2026-01-04 02:35:57,494 DEBUG Attempting to acquire lock 140503711745024 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/merges.txt.lock
2026-01-04 02:35:57,494 DEBUG Lock 140503711871424 acquired on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/model.safetensors.lock
2026-01-04 02:35:57,494 DEBUG Attempting to acquire lock 140503711748816 on pretrained_models/.cache/huggingface/download/README.md.lock
2026-01-04 02:35:57,500 DEBUG Lock 140503711866144 released on pretrained_models/.cache/h

✅ pretrained_models model already converted. You can find results in Fun-CosyVoice3-0.5B-2512-ov


PosixPath('Fun-CosyVoice3-0.5B-2512-ov')

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [4]:
from notebook_utils import device_widget

llm_device = device_widget("CPU", exclude=["AUTO"])

llm_device

Dropdown(description='Device:', options=('CPU', 'GPU'), value='CPU')

In [5]:
flow_device = device_widget("CPU", exclude=["NPU"])

flow_device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [6]:
hift_device = device_widget("CPU", exclude=["NPU"])

hift_device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [7]:
npu_ov_config = {
    "NPU_USE_NPUW": "YES",
    "NPUW_LLM": "YES",
    "NPUW_LLM_SHARED_HEAD": "False",
    "NPUW_ONLINE_PIPELINE": "NONE",
}

In [8]:
import sys

sys.path.append("CosyVoice/third_party/Matcha-TTS")
from ov_cosyvoice_helper import OVCosyVoice3


# Initialize OpenVINO CosyVoice3
# model_dir: OpenVINO model directory containing all converted models and dependency files
# If dependency files are not in ov_model_dir, provide original model_dir as second argument
cosyvoice = OVCosyVoice3(
    model_dir=model_path,
    device="CPU",  # Default device for all models
    llm_device=llm_device.value,  # LLM model device (defaults to device)
    flow_device=flow_device.value,  # Flow model device (defaults to device)
    hift_device=hift_device.value,  # HiFT model device (defaults to device)
    frontend_device="CPU",  # Frontend model device (defaults to device)
    npu_ov_config=npu_ov_config,
    hift_input_len=1000,
)

⌛ Loading OpenVINO frontend models on CPU...
⌛ Loading OpenVINO campplus model from Fun-CosyVoice3-0.5B-2512-ov/campplus.onnx...
✅ Campplus model loaded
⌛ Loading OpenVINO speech tokenizer model from Fun-CosyVoice3-0.5B-2512-ov/speech_tokenizer_v3.onnx...


2026-01-04 02:36:00,677 - modelscope - WARNING - Authentication has expired, please re-login with modelscope login --token "YOUR_SDK_TOKEN" if you need to access private models or datasets.
2026-01-04 02:36:00,680 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


✅ Speech tokenizer model loaded
failed to import ttsfrd, use wetext instead


2026-01-04 02:36:01,619 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 222
2026-01-04 02:36:01,858 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2026-01-04 02:36:01,958 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2026-01-04 02:36:02,895 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 222
2026-01-04 02:36:03,148 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None


✅ OpenVINO frontend loaded on CPU
⌛ Loading OpenVINO LLM models on CPU...
⌛ Loading OpenVINO models...
✅ Text embeddings model loaded
✅ Speech embeddings model loaded
✅ LLM model loaded
✅ OpenVINO LLM loaded on CPU
⌛ Loading OpenVINO Flow model on CPU...
⌛ Loading OpenVINO Flow embeddings model from Fun-CosyVoice3-0.5B-2512-ov/openvino_flow_embeddings_model.xml...
✅ Flow embeddings model loaded
⌛ Loading OpenVINO Flow estimator model from Fun-CosyVoice3-0.5B-2512-ov/openvino_flow_estimator_model.xml...
✅ Flow estimator model loaded
✅ OpenVINO Flow loaded on CPU
⌛ Loading OpenVINO HiFT model on CPU...
⌛ Loading OpenVINO HiFT model from Fun-CosyVoice3-0.5B-2512-ov/openvino_hift_model.xml...
✅ HiFT model loaded
✅ OpenVINO HiFT loaded on CPU


### Run zero_shot inference
[back to top ⬆️](#Table-of-contents:)

In [9]:
import torchaudio
import IPython


for i, j in enumerate(
    cosyvoice.inference_zero_shot(
        "Her handwriting is very neat, which suggests she likes things tidy.",
        "You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。",
        "./CosyVoice/asset/zero_shot_prompt.wav",
        stream=False,
    )
):
    torchaudio.save("ov_zero_shot_{}.wav".format(i), j["tts_speech"], cosyvoice.sample_rate)
    print(f"Saved ov_zero_shot_{i}.wav")
    display(IPython.display.Audio(f"ov_zero_shot_{i}.wav"))

  0%|                                                                                                          | 0/1 [00:00<?, ?it/s]2026-01-04 02:36:06,082 INFO synthesis text Her handwriting is very neat, which suggests she likes things tidy.


HiFT mel_input shape: (1, 80, 1000) (original: 228)


2026-01-04 02:36:08,931 INFO yield speech len 4.56, rtf 0.6248652935028076


Saved ov_zero_shot_0.wav


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.04s/it]


### Run cross_lingual inference with fine-grained control
[back to top ⬆️](#Table-of-contents:)

In [10]:
for i, j in enumerate(
    cosyvoice.inference_cross_lingual(
        "You are a helpful assistant.<|endofprompt|>[breath]Because people of their generation[breath]are more accustomed to living in the countryside,[breath]the neighbors are very friendly,[breath]um, and very familiar with each other.[breath]",
        "./CosyVoice/asset/zero_shot_prompt.wav",
        stream=False,
    )
):
    torchaudio.save("ov_fine_grained_control_{}.wav".format(i), j["tts_speech"], cosyvoice.sample_rate)
    print(f"Saved ov_fine_grained_control_{i}.wav")
    display(IPython.display.Audio(f"ov_fine_grained_control_{i}.wav"))

  0%|                                                                                                          | 0/1 [00:00<?, ?it/s]2026-01-04 02:36:08,996 INFO synthesis text You are a helpful assistant.<|endofprompt|>[breath]Because people of their generation[breath]are more accustomed to living in the countryside,[breath]the neighbors are very friendly,[breath]um, and very familiar with each other.[breath]


HiFT mel_input shape: (1, 80, 1000) (original: 480)


2026-01-04 02:36:13,640 INFO yield speech len 9.6, rtf 0.483796571691831


Saved ov_fine_grained_control_0.wav


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.70s/it]


### Run instruct2 inference (fast speed)
[back to top ⬆️](#Table-of-contents:)

In [11]:
for i, j in enumerate(
    cosyvoice.inference_instruct2(
        "Receiving a birthday gift from a friend far away, that unexpected surprise and deep blessing filled my heart with sweet happiness, and my smile bloomed like a flower.",
        "You are a helpful assistant. Please speak as fast as possible.<|endofprompt|>",
        "./CosyVoice/asset/zero_shot_prompt.wav",
        stream=False,
    )
):
    torchaudio.save("ov_instruct_fast_{}.wav".format(i), j["tts_speech"], cosyvoice.sample_rate)
    print(f"Saved ov_instruct_fast_{i}.wav")
    display(IPython.display.Audio(f"ov_instruct_fast_{i}.wav"))

  0%|                                                                                                          | 0/1 [00:00<?, ?it/s]2026-01-04 02:36:13,712 INFO synthesis text Receiving a birthday gift from a friend far away, that unexpected surprise and deep blessing filled my heart with sweet happiness, and my smile bloomed like a flower.


HiFT mel_input shape: (1, 80, 1000) (original: 448)


2026-01-04 02:36:18,227 INFO yield speech len 8.96, rtf 0.5038974806666374


Saved ov_instruct_fast_0.wav


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.58s/it]


### Run hotfix inference (fast speed)
[back to top ⬆️](#Table-of-contents:)

In [12]:
for i, j in enumerate(
    cosyvoice.inference_zero_shot(
        "Executives also praised the report through phone calls, text messages and other means.",
        "You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。",
        "./CosyVoice/asset/zero_shot_prompt.wav",
        stream=False,
    )
):
    torchaudio.save("ov_hotfix_{}.wav".format(i), j["tts_speech"], cosyvoice.sample_rate)
    print(f"Saved ov_hotfix_{i}.wav")
    display(IPython.display.Audio(f"ov_hotfix_{i}.wav"))

  0%|                                                                                                          | 0/1 [00:00<?, ?it/s]2026-01-04 02:36:18,287 INFO synthesis text Executives also praised the report through phone calls, text messages and other means.


HiFT mel_input shape: (1, 80, 1000) (original: 248)


2026-01-04 02:36:21,206 INFO yield speech len 4.96, rtf 0.5885205441905607


Saved ov_hotfix_0.wav


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.97s/it]


## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(cosyvoice)

# if you are launching remotely, specify server_name and server_port
#  demo.launch(server_name='your server name', server_port='server port in int')
# if you have any issue to launch on your platform, you can pass share=True to launch method:
# demo.launch(share=True)
# it creates a publicly shareable link for the interface. Read more in the docs: https://gradio.app/docs/
try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)